In [1]:
# Cell 1 — Read
from pyspark.sql import functions as F
from pyspark.sql import types as T

inv_df = spark.read \
    .option("header", "true") \
    .csv("abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/raw/inventory/inventory_snapshot.csv")

print("Raw inventory:", inv_df.count())


StatementMeta(, 677b4bda-2627-4b3c-a22a-284daa110c51, 3, Finished, Available, Finished, False)

Raw inventory: 610


In [2]:
# Cell 2 — Clean and write
cleaned_inv = inv_df \
    .withColumn("quantity_available", F.col("quantity_available").cast(T.IntegerType())) \
    .withColumn("quantity_reserved",  F.col("quantity_reserved").cast(T.IntegerType())) \
    .withColumn("reorder_threshold",  F.col("reorder_threshold").cast(T.IntegerType())) \
    .withColumn("last_updated",       F.to_timestamp(F.col("last_updated"))) \
    .withColumn(
        # Flag items below reorder level
        "needs_reorder",
        F.col("quantity_available") <= F.col("reorder_threshold")
    ) \
    .withColumn("silver_created_at", F.current_timestamp()) \
    .filter(F.col("product_id").isNotNull())

cleaned_inv.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_inventory")

print("Done:", spark.sql("SELECT COUNT(*) FROM silver_inventory").collect()[0][0])

StatementMeta(, 677b4bda-2627-4b3c-a22a-284daa110c51, 4, Finished, Available, Finished, False)

Done: 610
